# Stage 3 – Models

**Input:** `data/processed/loans_clean.parquet`.

**Output:** `data/processed/scores.parquet`, with the probability of default (PD) given by every model to every loan, and the fitted models in `models/`.

Three models are built on the loans issued in 2009-2011 and then frozen, as a bank would do:

1. a logistic regression, the classic scorecard;
2. XGBoost, the more complex challenger;
3. the Lending Club sub-grade, read as a third model "in production".

The models are the patient of this project, not its goal: they must be reasonable, not perfect.

In [1]:
import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from helpers import (DEV_YEARS, FEATURES, FIGURES, MODELS, PROCESSED, SEED,
                     calibration_table, gini, make_logit, make_xgb)

loans = pd.read_parquet(PROCESSED / "loans_clean.parquet")
print(f"{len(loans):,} loans")

211,505 loans


## 1. Development sample and holdout

The development sample is every loan issued in 2009-2011. A random 20% is set aside as a holdout: it shows how the models perform on loans they have not seen but that come from the same period. The quarters from 2012Q1 on are the monitoring period and are not touched here.

In [2]:
development = loans[loans["issue_year"].isin(DEV_YEARS)]
train, holdout = train_test_split(development, test_size=0.2, random_state=SEED,
                                  stratify=development["target"])

pd.DataFrame({
    "loans": [len(train), len(holdout)],
    "defaults": [train["target"].sum(), holdout["target"].sum()],
    "default_rate": [train["target"].mean(), holdout["target"].mean()],
}, index=["train", "holdout"]).round(4)

,loans,defaults,default_rate
train,22830,2578,0.1129
holdout,5708,644,0.1128


## 2. Logistic regression

The pipeline is defined in `helpers.make_logit`: median imputation with a "missing" flag, standard scaling, one-hot encoding, then the logistic regression. There are no class weights and no oversampling, because both would distort the predicted probabilities.

The numeric features are standardised, so each coefficient is the change in the log-odds of default for one standard deviation of the feature. The coefficient of a category is the difference from the most common answer (a mortgage, debt consolidation, income not verified).

In [3]:
logit = make_logit()
logit.fit(train[FEATURES], train["target"])

names = logit[:-1].get_feature_names_out()
coefficients = pd.Series(logit["model"].coef_[0], index=names).sort_values(key=abs, ascending=False)
coefficients.round(3).to_frame("coefficient").head(15)

,coefficient
purpose_small_business,0.977
home_ownership_OTHER,0.628
purpose_medical,0.433
purpose_renewable_energy,0.418
purpose_credit_card,-0.348
fico,-0.343
log_annual_inc,-0.329
purpose_moving,0.324
purpose_educational,0.286
purpose_other,0.267


## 3. XGBoost

Few hyperparameters and no long search: trees of depth 3 and a learning rate of 0.05. The only thing tuned is the number of trees, with early stopping: trees are added until the log-loss on the holdout has not improved for 50 rounds.

The final model is then a plain pipeline (`helpers.make_xgb`) with that number of trees.

In [4]:
search = make_xgb(n_trees=2000)
encode, booster = search["encode"], search["model"]
booster.set_params(early_stopping_rounds=50)

# Early stopping needs the holdout in the same encoded form as the training data.
encode.fit(train[FEATURES])
booster.fit(encode.transform(train[FEATURES]), train["target"],
            eval_set=[(encode.transform(holdout[FEATURES]), holdout["target"])], verbose=False)
n_trees = booster.best_iteration + 1
print("Trees chosen by early stopping:", n_trees)

xgb = make_xgb(n_trees)
xgb.fit(train[FEATURES], train["target"])

importance = pd.Series(xgb["model"].feature_importances_, index=xgb[:-1].get_feature_names_out())
importance.sort_values(ascending=False).round(3).to_frame("importance (gain)").head(10)

Trees chosen by early stopping: 293


,importance (gain)
fico,0.127
purpose_small_business,0.085
inq_last_6mths,0.079
purpose_credit_card,0.048
log_annual_inc,0.042
emp_length_years,0.039
pub_rec,0.036
revol_util,0.036
purpose_vacation,0.033
purpose_major_purchase,0.030


## 4. Benchmark: the Lending Club sub-grade

Lending Club gives every loan a sub-grade from A1 (safest) to G5 (riskiest). To compare it with the two models it needs to become a PD: the sub-grades are numbered from 1 to 35 and a one-variable logistic regression is fitted on the same training loans. The ranking is exactly the one of the sub-grade; the regression only attaches a probability to each step.

Unlike the two models above, the grade is not frozen: Lending Club kept updating its own grading. It shows what a maintained model looks like.

In [5]:
def sub_grade_number(sub_grade):
    # "A1" -> 1, "A5" -> 5, "B1" -> 6, ..., "G5" -> 35
    letter = sub_grade.str[0].map({letter: i for i, letter in enumerate("ABCDEFG")})
    return (letter * 5 + sub_grade.str[1].astype(int)).to_frame("sub_grade_number")


grade_model = LogisticRegression()
grade_model.fit(sub_grade_number(train["sub_grade"]), train["target"])

examples = pd.Series(["A1", "B1", "C1", "D1", "E1", "F1", "G1"])
example_pd = grade_model.predict_proba(sub_grade_number(examples))[:, 1]
pd.DataFrame([example_pd], columns=examples, index=["PD"]).round(3)

,A1,B1,C1,D1,E1,F1,G1
PD,0.06,0.084,0.118,0.162,0.22,0.29,0.373


## 5. Performance on the development sample

AUC and Gini on the training loans and on the holdout. A large gap between the two would mean overfitting. The holdout Gini is the reference for the traffic lights of stage 4.

XGBoost has a small advantage on the holdout, because the holdout was used to choose its number of trees.

In [6]:
def predict_all(data):
    return pd.DataFrame({
        "pd_logit": logit.predict_proba(data[FEATURES])[:, 1],
        "pd_xgb": xgb.predict_proba(data[FEATURES])[:, 1],
        "pd_grade": grade_model.predict_proba(sub_grade_number(data["sub_grade"]))[:, 1],
    }, index=data.index)


performance = []
for name, data in [("train", train), ("holdout", holdout)]:
    predictions = predict_all(data)
    for model in predictions.columns:
        performance.append({
            "sample": name, "model": model,
            "auc": roc_auc_score(data["target"], predictions[model]),
            "gini": gini(data["target"], predictions[model]),
            "mean_pd": predictions[model].mean(),
            "default_rate": data["target"].mean(),
        })
performance = pd.DataFrame(performance).set_index(["sample", "model"])
performance.round(4)

auc    gini  mean_pd  default_rate
sample  model                                          
train   pd_logit  0.6886  0.3772   0.1129        0.1129
        pd_xgb    0.7574  0.5148   0.1129        0.1129
        pd_grade  0.6379  0.2758   0.1129        0.1129
holdout pd_logit  0.6943  0.3887   0.1131        0.1128
        pd_xgb    0.7036  0.4072   0.1134        0.1128
        pd_grade  0.6562  0.3125   0.1120        0.1128

**Calibration by decile on the holdout.** Loans are sorted by PD and cut into ten groups: the mean PD of each group should be close to its observed default rate.

In [7]:
holdout_predictions = predict_all(holdout)
calibration = pd.concat(
    {model: calibration_table(holdout["target"], holdout_predictions[model]) for model in ["pd_logit", "pd_xgb"]},
    axis=1,
)
calibration.round(3)

pd_logit                      pd_xgb                     
          loans mean_pd default_rate  loans mean_pd default_rate
bucket                                                          
1           571   0.031        0.033    571   0.027        0.021
2           571   0.049        0.042    571   0.047        0.049
3           571   0.062        0.042    571   0.063        0.051
4           570   0.076        0.079    570   0.077        0.068
5           571   0.090        0.082    571   0.091        0.077
6           571   0.105        0.107    571   0.106        0.117
7           570   0.123        0.139    570   0.124        0.126
8           571   0.145        0.163    571   0.147        0.154
9           571   0.178        0.180    571   0.180        0.193
10          571   0.272        0.261    571   0.272        0.271

## 6. Score every loan

The three frozen models are applied to all the loans, development and monitoring. The column `sample` says where each loan comes from, so that stage 4 can use the holdout as the reference.

In [8]:
scores = loans[["id", "issue_quarter", "issue_year", "target", "grade", "sub_grade"]].copy()
scores["sample"] = "monitoring"
scores.loc[train.index, "sample"] = "train"
scores.loc[holdout.index, "sample"] = "holdout"
scores = scores.join(predict_all(loans))

scores.to_parquet(PROCESSED / "scores.parquet", index=False)
scores.groupby("sample")[["target", "pd_logit", "pd_xgb", "pd_grade"]].mean().round(4)

,target,pd_logit,pd_xgb,pd_grade
sample,,,,
holdout,0.1128,0.1131,0.1134,0.1120
monitoring,0.1358,0.1289,0.1263,0.1169
train,0.1129,0.1129,0.1129,0.1129


## 7. Save the models

Stage 5 loads them again to study how they use the features.

In [9]:
joblib.dump(logit, MODELS / "logit.joblib")
joblib.dump(xgb, MODELS / "xgb.joblib")
joblib.dump(grade_model, MODELS / "grade_model.joblib")
performance.to_csv(FIGURES / "table_development_performance.csv")
print("Saved:", sorted(path.name for path in MODELS.glob("*.joblib")))

Saved: ['grade_model.joblib', 'logit.joblib', 'xgb.joblib']
